# PCA

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

from sklearn.datasets import load_wine, load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

wine = load_wine()

x = wine.data
y = wine.target
feat_names = wine.feature_names
target_names = wine.target_names

print("Original dataset shape:", x.shape)
print("Target shape:", y.shape)
print("Feature names:", feat_names)
print("Target names:", target_names)

# Step1: Standardize the data
scaler = StandardScaler()
x_scaled = scaler.fit_transform(x)

# Step2: Apply PCA
pca = PCA(n_components=2)
x_reduced = pca.fit_transform(x_scaled)

print("Original dataset shape:", x.shape)
print("Reduced dataset shape:", x_reduced.shape)

wine_colors = ['red', 'green', 'blue']

plt.figure(figsize=(10, 6))

# Fixed: changed 'sip' to 'zip' and used 'target_name' as a loop variable to prevent shadowing
for class_num, color, target_name in zip(
    [0, 1, 2], wine_colors, wine.target_names
):
    plt.scatter(
        x_reduced[y == class_num, 0],
        x_reduced[y == class_num, 1],
        color=color,
        label=target_name,
        alpha=0.7
    )

plt.title("PCA of Wine Dataset")
plt.xlabel("First Principal Component")
plt.ylabel("Second Principal Component")
plt.legend()
plt.grid(alpha=0.2)
plt.show()

print("Principal component loadings;")
print(pca.components_)

# Fixed: Corrected 'Pca' to lowercase 'pca'
print("Shape of principal components: ", pca.components_.shape)

plt.figure(figsize=(14, 5))

sns.heatmap(
    pca.components_,
    cmap='viridis',
    xticklabels=feat_names,
    yticklabels=['PC1', 'PC2']
)

plt.title("Heatmap of Principal Components")
plt.xlabel("Features")
plt.ylabel("Principal Components")
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

print("Explained variance:")
print(pca.explained_variance_)

print("\nExplained variance ratio:")
print(pca.explained_variance_ratio_)


pca_3 = PCA(n_components=0.96)
x_reduced_96 = pca_3.fit_transform(x_scaled)

print("Number of components selected:", pca_3.n_components_)
print("Total explained varience:",
      np.sum(pca_3.explained_variance_ratio_))

print("Reduced dataset shape: ", X_reduced_96.shape)

# RANDOM FOREST

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score

# Step 1: Load the dataset
url = (
    "https://archive.ics.uci.edu/ml/"
    "machine-learning-databases/cpu-performance/machine.data"
)

columns = [
    "vendor", "model", "MYCT", "MMIN", "MMAX",
    "CACH", "CHMIN", "CHMAX", "PRP", "ERP"
]

data = pd.read_csv(url, names=columns)

# Step 2: Display the dataset
print("First five rows:")
print(data.head())

print("\nDataset shape:", data.shape)

print("\nMissing values:")
print(data.isnull().sum())

# Step 3: Display histograms
data.hist(figsize=(12, 8), bins=15)
plt.tight_layout()
plt.show()

# Step 4: Correlation between numerical columns
correlation = data.select_dtypes(include=np.number).corr()

print("\nCorrelation with PRP:")
print(correlation["PRP"].sort_values(ascending=False))

# Display correlation heatmap
plt.figure(figsize=(10, 7))
sns.heatmap(
    correlation,
    annot=True,
    cmap="coolwarm",
    fmt=".2f"
)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

# Step 5: Select features and target
X = data.drop(columns=["PRP", "ERP", "vendor", "model"])
y = data["PRP"]

print("\nFeatures:")
print(X.head())

print("\nTarget:")
print(y.head())

# Step 6: Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("\nTraining samples:", len(X_train))
print("Testing samples:", len(X_test))

# Step 7: Create the Random Forest model
forest_reg = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

# Step 8: Train the model
forest_reg.fit(X_train, y_train)

# Step 9: Predict training and testing results
y_train_pred = forest_reg.predict(X_train)
y_test_pred = forest_reg.predict(X_test)

# Step 10: Evaluate training performance
train_rmse = np.sqrt(
    mean_squared_error(y_train, y_train_pred)
)
train_r2 = r2_score(y_train, y_train_pred)

print("\nTraining Results")
print("Training RMSE:", round(train_rmse, 2))
print("Training R2 Score:", round(train_r2, 4))

# Step 11: Evaluate testing performance
test_rmse = np.sqrt(
    mean_squared_error(y_test, y_test_pred)
)
test_r2 = r2_score(y_test, y_test_pred)

print("\nTesting Results")
print("Testing RMSE:", round(test_rmse, 2))
print("Testing R2 Score:", round(test_r2, 4))

# Step 12: Plot actual vs predicted values
plt.figure(figsize=(8, 6))

plt.scatter(y_test, y_test_pred, alpha=0.7)

# Reference line for perfect predictions
min_value = min(y_test.min(), y_test_pred.min())
max_value = max(y_test.max(), y_test_pred.max())

plt.plot(
    [min_value, max_value],
    [min_value, max_value],
    "r--"
)

plt.xlabel("Actual PRP")
plt.ylabel("Predicted PRP")
plt.title("Actual vs Predicted CPU Performance")
plt.tight_layout()
plt.show()